In [9]:
import importlib
import pandas as pd
import numpy as np

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

import lib.lib as lib
importlib.reload(lib)

from lib.lib import *


In [11]:
# =========================================================
# 1. 설정
# =========================================================

ETF_CODE = "SMH"

N_DAYS = 5
THRESHOLD = 0.01          # 예: 5일 뒤 +1% 이상이면 1
PRED_THRESHOLD = 0.5

RANDOM_STATE = 42

EXCEL_PATH = "./experiments_excel/experiment_summary2.xlsx"
SHEET_NAME = 0

DATE_COL = "Date"


# =========================================================
# 2. 엑셀에서 feature / lag 정보 로드
# =========================================================

feature_info_df = pd.read_excel(EXCEL_PATH, sheet_name=SHEET_NAME)

print("feature_info_df shape:", feature_info_df.shape)
display(feature_info_df.head())


# =========================================================
# 3. 엑셀 컬럼명 정리
# - 네가 만든 엑셀에 따라 feature / lag 컬럼명을 맞춰줌
# =========================================================

feature_info_df = feature_info_df.copy()

# feature명 컬럼 자동 인식
if "feature" in feature_info_df.columns:
    feature_col = "feature"
elif "feature_nm" in feature_info_df.columns:
    feature_col = "feature_nm"
elif "base_feature" in feature_info_df.columns:
    feature_col = "base_feature"
else:
    raise ValueError("엑셀에 feature / feature_nm / base_feature 중 하나가 있어야 합니다.")

# lag 컬럼 자동 인식
if "lag" in feature_info_df.columns:
    lag_col = "lag"
elif "best_lag" in feature_info_df.columns:
    lag_col = "best_lag"
elif "selected_lag" in feature_info_df.columns:
    lag_col = "selected_lag"
else:
    raise ValueError("엑셀에 lag / best_lag / selected_lag 중 하나가 있어야 합니다.")

selected_lag_df = feature_info_df[[feature_col, lag_col]].copy()
selected_lag_df.columns = ["feature", "lag"]

# lag 값 정리: lag0, lag3, lagm3 같은 문자열 대응
def clean_lag_value(x):
    if pd.isna(x):
        return np.nan

    if isinstance(x, str):
        x = x.strip()
        x = x.replace("lagm", "-")
        x = x.replace("lag", "")
        return int(float(x))

    return int(x)

selected_lag_df["lag"] = selected_lag_df["lag"].apply(clean_lag_value)
selected_lag_df = selected_lag_df.dropna(subset=["feature", "lag"]).copy()
selected_lag_df["lag"] = selected_lag_df["lag"].astype(int)

print("사용 feature 수:", len(selected_lag_df))
display(selected_lag_df.head())


# =========================================================
# 4. base_df 준비
# - 이미 base_df가 메모리에 있으면 이 셀은 건너뛰어도 됨
# - 여기서는 기존 lib.py 방식 기준
# =========================================================

external_tickers = {
    "QQQ": "QQQ",
    "SPY": "SPY",
    "SOXX": "SOXX",
    "NVDA": "NVDA",
    "TSM": "TSM",
    "VIX": "^VIX",
    "TNX": "^TNX",
    "KRW": "KRW=X",
    "DXY": "DX-Y.NYB",
    "GOLD": "GC=F",
    "OIL": "CL=F",
}

external_feature_types = {
    "QQQ": "price",
    "SPY": "price",
    "SOXX": "price",
    "NVDA": "price",
    "TSM": "price",
    "VIX": "risk",
    "TNX": "rate",
    "KRW": "price",
    "DXY": "price",
    "GOLD": "price",
    "OIL": "price",
}

base_df, feature_cols, close_col = make_base_feature_dataset(
    etf_code=ETF_CODE,
    external_tickers=external_tickers,
    external_feature_types=external_feature_types,
    start_date="2020-01-01",
    end_date=None,
)

print("base_df shape:", base_df.shape)
print("close_col:", close_col)
print("기간:", base_df[DATE_COL].min(), "~", base_df[DATE_COL].max())


# =========================================================
# 5. target 생성
# - 주의: 최근 N_DAYS개는 미래 수익률을 모르므로 target이 NaN
# =========================================================

target_df, target_col = add_target_column(
    df=base_df,
    close_col=close_col,
    n_days=N_DAYS,
    threshold=THRESHOLD,
)

print("target_col:", target_col)
print("target_df shape:", target_df.shape)
print("마지막 날짜:", target_df[DATE_COL].max())

display(target_df.tail(10)[[DATE_COL, close_col, f"future_ret_{N_DAYS}d", target_col]])


# =========================================================
# 6. 엑셀 feature 기준으로 lagged dataset 생성
# - 여기서는 예측 최신 row를 살려야 하므로 dropna를 마지막에 함부로 하지 않음
# =========================================================

model_df = pd.DataFrame()
model_df[DATE_COL] = target_df[DATE_COL]
model_df[close_col] = target_df[close_col]
model_df[f"future_ret_{N_DAYS}d"] = target_df[f"future_ret_{N_DAYS}d"]
model_df[target_col] = target_df[target_col]

lagged_feature_cols = []

for _, row in selected_lag_df.iterrows():
    feature = row["feature"]
    lag = int(row["lag"])

    if feature not in target_df.columns:
        print(f"[SKIP] base_df에 없는 feature: {feature}")
        continue

    lagged_col = f"{feature}_lag{lag}"
    model_df[lagged_col] = target_df[feature].shift(lag)
    lagged_feature_cols.append(lagged_col)

model_df = model_df.replace([np.inf, -np.inf], np.nan)

print("model_df shape:", model_df.shape)
print("lagged feature 수:", len(lagged_feature_cols))

display(model_df.tail(10))


# =========================================================
# 7. 가장 최근 날짜를 예측 대상으로 분리
# =========================================================

predict_date = model_df[DATE_COL].max()

predict_df = model_df[model_df[DATE_COL] == predict_date].copy()

# 학습 데이터:
# 1) 예측일보다 이전 날짜
# 2) target 존재
# 3) feature 결측 없음
train_df = model_df[
    (model_df[DATE_COL] < predict_date) &
    (model_df[target_col].notna())
].copy()

train_df = train_df.dropna(subset=lagged_feature_cols + [target_col]).copy()

print("예측 대상일:", predict_date)
print("train_df 기간:", train_df[DATE_COL].min(), "~", train_df[DATE_COL].max())
print("train_df shape:", train_df.shape)
print("predict_df shape:", predict_df.shape)

print()
print("train target 분포:")
print(train_df[target_col].value_counts())
print(train_df[target_col].value_counts(normalize=True))


# =========================================================
# 8. 예측 row 사용 가능 여부 확인
# =========================================================

missing_pred_cols = [
    col for col in lagged_feature_cols
    if predict_df[col].isna().any()
]

if len(missing_pred_cols) > 0:
    print("예측 row에서 결측인 feature 수:", len(missing_pred_cols))
    print(missing_pred_cols[:20])
    raise ValueError("최신 날짜 예측 row에 결측 feature가 있습니다. lag / 외부지표 ffill 여부 확인 필요.")

X_train = train_df[lagged_feature_cols].copy()
y_train = train_df[target_col].astype(int).copy()

X_pred = predict_df[lagged_feature_cols].copy()

print("X_train shape:", X_train.shape)
print("X_pred shape:", X_pred.shape)


# =========================================================
# 9. 모델 학습
# =========================================================

model = RandomForestClassifier(
    n_estimators=500,
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    max_features="sqrt",
    class_weight="balanced",
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

model.fit(X_train, y_train)

print("모델 학습 완료")


# =========================================================
# 10. in-sample 성능 확인
# - 이건 검증 성능이 아니라 학습 데이터 기준 참고용
# =========================================================

train_pred = model.predict(X_train)
train_proba = model.predict_proba(X_train)[:, 1]

print("accuracy :", accuracy_score(y_train, train_pred))
print("precision:", precision_score(y_train, train_pred, zero_division=0))
print("recall   :", recall_score(y_train, train_pred, zero_division=0))
print("f1       :", f1_score(y_train, train_pred, zero_division=0))
print()
print("confusion matrix:")
print(confusion_matrix(y_train, train_pred))


# =========================================================
# 11. 최신 날짜 예측
# =========================================================

pred_proba = model.predict_proba(X_pred)[:, 1]
pred_label = (pred_proba >= PRED_THRESHOLD).astype(int)

prediction_result_df = predict_df[[DATE_COL, close_col]].copy()
prediction_result_df["pred_proba_1"] = pred_proba
prediction_result_df["pred_label"] = pred_label
prediction_result_df["pred_threshold"] = PRED_THRESHOLD
prediction_result_df["n_days"] = N_DAYS
prediction_result_df["target_threshold"] = THRESHOLD
prediction_result_df["target_col"] = target_col

display(prediction_result_df)

print("예측 완료")

feature_info_df shape: (13, 80)


,experiment_key,created_dt,run_id,etf_code,rolling_precision_mean,n_days,threshold,valid_months,vif_threshold,lag_search_years,...,rolling_fp_max,rolling_tn_mean,rolling_tn_std,rolling_tn_min,rolling_tn_max,rolling_fn_mean,rolling_fn_std,rolling_fn_min,rolling_fn_max,rolling_pred_1_precision_zero_fill_mean
0,c10f41c740d58cdf50d90f1414d63978,2026-05-23 18:39:22,20260523_183922_optuna_trial_0011,SMH,0.655761,5,0.005,3,10,1,...,25,10.9,4.357624,3,17,14.1,9.085397,3,26,0.655761
1,30d0a7c2f12600ac675b60aca4480ec5,2026-05-23 18:30:47,20260523_183047_optuna_trial_0009,SMH,0.628678,10,0.005,3,10,1,...,19,6.7,2.496664,3,11,16.3,8.692909,3,29,0.628678
2,4500a5c35a61c38f51d858ceaf709f31,2026-05-23 17:25:42,20260523_172542_optuna_trial_0000,SMH,0.625037,5,0.005,3,10,1,...,28,5.3,3.683296,0,12,6.7,7.071853,0,18,0.625037
3,8610ec25bea1d0a1cc40ff2bca815dff,2026-05-23 18:34:33,20260523_183433_optuna_trial_0010,SMH,0.610649,10,0.005,3,10,1,...,18,10.4,4.575296,4,18,26.3,9.177872,13,41,0.610649
4,e1bbf3156de6a44229c7ae594395c7e7,2026-05-23 17:37:05,20260523_173705_optuna_trial_0004,SMH,0.575062,5,0.010,3,10,1,...,29,0.2,0.632456,0,2,0.0,0.000000,0,0,0.575062


ValueError: 엑셀에 feature / feature_nm / base_feature 중 하나가 있어야 합니다.

In [ ]:
# # =========================================================
# # 1. target 없는 base dataset 생성
# # =========================================================
# print()
# print("=" * 50)
# print("1. Base feature dataset created.")

# base_df, base_feature_cols, close_col = make_base_feature_dataset(
#     etf_code=ETF_CODE,
#     external_tickers=EXTERNAL_TICKERS,
#     external_feature_types=EXTERNAL_FEATURE_TYPES,
#     start_date=START_DATE,
#     end_date=END_DATE
# )

# max_date = base_df["Date"].max()
# valid_start_date = max_date - pd.DateOffset(months=VALID_MONTHS)

# train_base_df = base_df[
#     base_df["Date"] < valid_start_date
# ].copy()

# valid_base_df = base_df[
#     base_df["Date"] >= valid_start_date
# ].copy()

# print("train_base_df shape:", train_base_df.shape)
# print("valid_base_df shape:", valid_base_df.shape)
# print("close_col:", close_col)
# print("=" * 50)

# # =========================================================
# # 2. VIF 기반 불필요 칼럼 제거
# # =========================================================
# print()
# print("=" * 50)
# print("2. VIF filtering completed.")

# vif_feature_cols, removed_vif_df, final_vif_df = reduce_features_by_vif(
#     df=train_base_df,
#     feature_cols=base_feature_cols,
#     vif_threshold=VIF_THRESHOLD,
#     date_col="Date",
#     verbose=True
# )

# # VIF 통과 feature만 남긴 데이터셋 생성
# keep_cols = ["Date", close_col] + vif_feature_cols
# vif_filtered_df = train_base_df[keep_cols].copy()

# print("vif_filtered_df shape:", vif_filtered_df.shape)
# print("제거된 컬럼:")
# print(removed_vif_df)
# print("=" * 50)

# # =========================================================
# # 3. Target 변수 생성
# # =========================================================
# print()
# print("=" * 50)
# print("3. Target column added.")

# target_df, target_col = add_target_column(
#     df=vif_filtered_df,
#     close_col=close_col,
#     n_days=N_DAYS,
#     threshold=THRESHOLD
# )

# print("target_col:", target_col)

# # target 없는 마지막 n_days 행 제거
# target_df = target_df.dropna(subset=[target_col]).copy()
# target_df[target_col] = target_df[target_col].astype(int)

# print("target_df shape after dropna:", target_df.shape)
# print("target 분포:")
# print(target_df[target_col].value_counts())
# print("target 비율:")
# print(target_df[target_col].value_counts(normalize=True))

# print("=" * 50)

# # =========================================================
# # 4.1. 최근 1년 데이터만 사용해서 lag 탐색
# # =========================================================
# print()
# print("=" * 50)
# print("4. 변수별 최적 LAG 탐색 완료.")

# max_date = target_df["Date"].max()
# lag_search_start_date = max_date - pd.DateOffset(years=LAG_SEARCH_YEARS)

# target_df_for_lag_search = target_df[
#     target_df["Date"] >= lag_search_start_date
# ].copy()

# print("lag 탐색 기준 기간:")
# print(target_df_for_lag_search["Date"].min(), "~", target_df_for_lag_search["Date"].max())
# print("lag 탐색용 데이터 shape:", target_df_for_lag_search.shape)


# # =========================================================
# # 4.2. lag 탐색 실행
# # =========================================================
# exclude_cols_for_lag = [
#     "Date",
#     close_col,
#     f"future_ret_{N_DAYS}d",
#     target_col
# ]

# lag_search_feature_cols = [
#     col for col in target_df.columns
#     if col not in exclude_cols_for_lag
# ]

# lag_result_df, best_lag_df = find_best_lag_by_feature(
#     df=target_df_for_lag_search,   # 핵심: 전체 target_df 말고 최근 1년만 넣음
#     feature_cols=lag_search_feature_cols,
#     target_col=target_col,
#     lag_days=LAG_DAYS,
#     date_col="Date"
# )

# print("전체 lag 탐색 결과 shape:", lag_result_df.shape)

# # =========================================================
# # 4.3. best lag 적용해서 lagged_df 생성
# # =========================================================

# lagged_df, lagged_feature_cols = make_lagged_dataset_by_best_lag(
#     df=target_df,
#     best_lag_df=best_lag_df,
#     target_col=target_col,
#     close_col=close_col,
#     n_days=N_DAYS,
#     date_col="Date"
# )

# print("lagged_df shape:", lagged_df.shape)
# print("lagged feature count:", len(lagged_feature_cols))

# print("=" * 50)


# # =========================================================
# # # 5.1. permutation importance 실행
# # =========================================================
# print()
# print("=" * 50)
# print("5. RandomForest in-sample 학습 + permutation importance 완료.")

# importance_df, raw_importance_df, baseline_df = run_rf_permutation_importance_in_sample(
#     lagged_df=lagged_df,
#     feature_cols=lagged_feature_cols,
#     target_col=target_col,
#     date_col="Date",
#     close_col=close_col,
#     n_rf_runs=N_RF_RUNS,
#     n_repeats=N_REPEATS,
#     random_state=RANDOM_STATE
# )


# # =========================================================
# # 5.2. 결과 feature명 / lag 분리해서 보기
# # =========================================================

# importance_view_df = importance_df.copy()

# importance_view_df[["base_feature", "selected_lag"]] = importance_view_df["feature"].apply(
#     lambda x: pd.Series(split_lagged_feature_name(x))
# )

# importance_view_df = importance_view_df[
#     [
#         "feature",
#         "base_feature",
#         "selected_lag",
#         "importance_score",
#         "importance_mean",
#         "importance_std",
#         "importance_var",
#         "importance_min",
#         "importance_max",
#         "run_count",
#         "repeat_count"
#     ]
# ]

# # =========================================================
# # 5.3. best_lag_df와 importance 결과 합치기
# # =========================================================

# importance_with_lag_df = importance_view_df.merge(
#     best_lag_df.rename(columns={
#         "feature": "base_feature",
#         "lag": "best_lag",
#         "corr": "lag_corr",
#         "abs_corr": "lag_abs_corr",
#         "n_rows": "lag_n_rows"
#     }),
#     on="base_feature",
#     how="left"
# )

# importance_with_lag_df = importance_with_lag_df[
#     [
#         "feature",
#         "base_feature",
#         "selected_lag",
#         "importance_score",
#         "importance_mean",
#         "importance_std",
#         "importance_var",
#         "importance_min",
#         "importance_max",
#         "best_lag",
#         "lag_corr",
#         "lag_abs_corr",
#         "lag_n_rows",
#         "run_count",
#         "repeat_count"
#     ]
# ]

# # 보기 좋게 정렬
# importance_with_lag_df = importance_with_lag_df.sort_values(
#     "importance_score",
#     ascending=False
# ).reset_index(drop=True)

# # =========================================================
# # 5.4. TOP 변수 추출
# # =========================================================

# top_feature_df = importance_with_lag_df.head(TOP_N).copy()
# top_feature_cols = top_feature_df["feature"].tolist()

# print("TOP feature count:", len(top_feature_cols))
# print(top_feature_cols)
# print("=" * 50)



1. Base feature dataset created.
Loading external ticker: QQQ / QQQ
Loading external ticker: SPY / SPY
Loading external ticker: SOXX / SOXX
Loading external ticker: NVDA / NVDA
Loading external ticker: TSM / TSM
Loading external ticker: VIX / ^VIX
Loading external ticker: TNX / ^TNX
Loading external ticker: USDKRW / KRW=X
Loading external ticker: DXY / DX-Y.NYB
Loading external ticker: GOLD / GC=F
Loading external ticker: OIL / CL=F
train_base_df shape: (1584, 34)
valid_base_df shape: (23, 34)
close_col: SMH_adj_close

2. VIF filtering completed.
VIF 계산 대상 row 수: 1525
VIF 계산 대상 feature 수: 32
현재 max VIF: 156.57 / feature: SMH_ret_5d
현재 max VIF: 114.62 / feature: SMH_ret_20d
현재 max VIF: 15.18 / feature: QQQ_ret_20d
현재 max VIF: 12.16 / feature: QQQ_ret_5d
현재 max VIF: 10.20 / feature: SOXX_ret_20d
현재 max VIF: 9.37 / feature: SOXX_ret_5d

========== VIF 제거 결과 ==========
초기 feature 수: 32
상수 제거 feature 수: 0
VIF 제거 feature 수: 5
최종 feature 수: 27
vif_filtered_df shape: (1584, 29)
제거된 컬럼:
  remo

In [8]:
# =========================================================
# 6. valid_df 생성
# - base_df 전체에서 top_feature_df 기준 lag feature 생성
# - 마지막에 valid_base_df 길이만큼만 자름
# - 여기서는 dropna 절대 하지 않음
# - X, y 생성하지 않음
# =========================================================

print("=" * 60)
print("6. valid_df 생성")
print("=" * 60)

# ---------------------------------------------------------
# 1. top_feature_df 준비
# ---------------------------------------------------------

top_feature_df = top_feature_df.copy()

if ("base_feature" not in top_feature_df.columns) or ("selected_lag" not in top_feature_df.columns):
    top_feature_df[["base_feature", "selected_lag"]] = top_feature_df["feature"].apply(
        lambda x: pd.Series(split_lagged_feature_name(x))
    )

top_feature_df["selected_lag"] = top_feature_df["selected_lag"].astype(int)

top_feature_cols = top_feature_df["feature"].tolist()
top_base_features = top_feature_df["base_feature"].unique().tolist()

print("top feature 수:", len(top_feature_cols))

# ---------------------------------------------------------
# 2. base_df에서 필요한 컬럼만 필터링
# ---------------------------------------------------------

need_cols = ["Date", close_col] + top_base_features

missing_cols = [col for col in need_cols if col not in base_df.columns]

if len(missing_cols) > 0:
    raise ValueError(f"base_df에 없는 컬럼이 있습니다: {missing_cols}")

valid_df = base_df[need_cols].copy()
valid_df = valid_df.sort_values("Date").reset_index(drop=True)

# ---------------------------------------------------------
# 3. 전체 데이터 기준으로 lag feature 생성
# ---------------------------------------------------------

for _, row in top_feature_df.iterrows():
    base_feature = row["base_feature"]
    selected_lag = int(row["selected_lag"])
    lagged_feature = row["feature"]

    valid_df[lagged_feature] = valid_df[base_feature].shift(selected_lag)

print("lag 생성 후 valid_df shape:", valid_df.shape)


# ---------------------------------------------------------
# 4. lag 생성 전 원본 base_feature 컬럼 제거
#    Date, close_col, top_feature_cols만 남김
# ---------------------------------------------------------

valid_df = valid_df[["Date", close_col] + top_feature_cols].copy()

print("원본 base_feature 제거 후 valid_df shape:", valid_df.shape)
print("최종 컬럼 수:", len(valid_df.columns))


# ---------------------------------------------------------
# 5. target 생성
# ---------------------------------------------------------

valid_df, _ = add_target_column(
    df=valid_df,
    close_col=close_col,
    n_days=N_DAYS,
    threshold=THRESHOLD,
    target_col=target_col
)

print("target 생성 후 valid_df shape:", valid_df.shape)


# ---------------------------------------------------------
# 6. 최근 valid_base_df 길이만큼 자르기
# ---------------------------------------------------------

valid_df = valid_df.tail(len(valid_base_df)).copy()
valid_df = valid_df.reset_index(drop=True)

print("최종 valid_df shape:", valid_df.shape)
print("valid_base_df shape:", valid_base_df.shape)

print("valid_df 기간:")
print(valid_df["Date"].min(), "~", valid_df["Date"].max())

# =========================================================
# 7. Train 학습 후 valid_df 예측
# =========================================================

print("=" * 60)
print("7. Train 학습 후 valid_df 예측")
print("=" * 60)

# ---------------------------------------------------------
# 1. top feature 컬럼 확인
# ---------------------------------------------------------

top_feature_cols = top_feature_df["feature"].tolist()

missing_train_cols = [col for col in top_feature_cols if col not in lagged_df.columns]
missing_valid_cols = [col for col in top_feature_cols if col not in valid_df.columns]

if len(missing_train_cols) > 0:
    raise ValueError(f"lagged_df에 없는 top feature가 있습니다: {missing_train_cols}")

if len(missing_valid_cols) > 0:
    raise ValueError(f"valid_df에 없는 top feature가 있습니다: {missing_valid_cols}")

print("사용 feature 수:", len(top_feature_cols))


# ---------------------------------------------------------
# 2. Train 데이터 준비
# ---------------------------------------------------------

train_df = lagged_df[
    ["Date", close_col, target_col] + top_feature_cols
].copy()

train_df = train_df.replace([np.inf, -np.inf], np.nan)
train_df = train_df.dropna(subset=top_feature_cols + [target_col]).copy()

X_train = train_df[top_feature_cols].copy()
y_train = train_df[target_col].astype(int).copy()

print("train_df shape:", train_df.shape)
print("X_train shape:", X_train.shape)

print("train target 분포:")
print(y_train.value_counts())
print(y_train.value_counts(normalize=True))


# ---------------------------------------------------------
# 3. valid_df 예측용 데이터 준비
#    valid_df 행 수는 유지
#    단, 모델 입력용 X_valid_temp에는 결측 없는 행만 사용
# ---------------------------------------------------------

valid_pred_df = valid_df.copy()

valid_pred_df["pred_proba"] = np.nan
valid_pred_df["pred"] = np.nan

valid_available_mask = (
    valid_pred_df[top_feature_cols]
    .replace([np.inf, -np.inf], np.nan)
    .notna()
    .all(axis=1)
)

X_valid = valid_pred_df.loc[valid_available_mask, top_feature_cols].copy()

print("valid_df shape:", valid_df.shape)
print("예측 가능한 valid row 수:", len(X_valid))
print("예측 불가능 row 수:", len(valid_df) - len(X_valid))


# ---------------------------------------------------------
# 4. RandomForest 학습
# ---------------------------------------------------------

rf = RandomForestClassifier(
    n_estimators=500,
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=1,
    max_features="sqrt",
    class_weight="balanced",
    random_state=RANDOM_STATE,
    n_jobs=-1
)

rf.fit(X_train, y_train)

print("모델 학습 완료")


# ---------------------------------------------------------
# 5. valid_df 예측
# ---------------------------------------------------------


valid_pred_proba = rf.predict_proba(X_valid)[:, 1]
valid_pred = (valid_pred_proba >= PRED_THRESHOLD).astype(int)

valid_pred_df.loc[valid_available_mask, "pred_proba"] = valid_pred_proba
valid_pred_df.loc[valid_available_mask, "pred"] = valid_pred

valid_pred_df["pred"] = valid_pred_df["pred"].astype("Int64")

print("valid_df 예측 완료")


# ---------------------------------------------------------
# 6. 예측 결과 확인
# ---------------------------------------------------------

display_cols = ["Date", close_col, "pred_proba", "pred"]

# target_col이 valid_df에 있으면 같이 보기
if target_col in valid_pred_df.columns:
    display_cols = ["Date", close_col, target_col, "pred_proba", "pred"]

display(valid_pred_df[display_cols])

print("예측값 분포:")
print(valid_pred_df["pred"].value_counts(dropna=False))

# =========================================================
# 7. 예측 1 기준 정확도 확인
# - pred = 1 인 것 중 실제 target = 1 인 비율
# - 우리가 주로 볼 precision 지표
# =========================================================

print("=" * 60)
print("7. 예측 1 기준 정확도 확인")
print("=" * 60)

# ---------------------------------------------------------
# 1. 평가 가능한 row만 사용
# - pred가 있어야 함
# - target도 있어야 함
# ---------------------------------------------------------

eval_df = valid_pred_df[
    valid_pred_df["pred"].notna() &
    valid_pred_df[target_col].notna()
].copy()

eval_df["pred"] = eval_df["pred"].astype(int)
eval_df[target_col] = eval_df[target_col].astype(int)

print("평가 가능 row 수:", len(eval_df))

# ---------------------------------------------------------
# 2. pred = 1인 row만 필터링
# ---------------------------------------------------------

pred_1_df = eval_df[eval_df["pred"] == 1].copy()

pred_1_count = len(pred_1_df)
pred_1_actual_1_count = (pred_1_df[target_col] == 1).sum()

print("예측 1 개수:", pred_1_count)
print("예측 1 중 실제 1 개수:", pred_1_actual_1_count)

# ---------------------------------------------------------
# 3. 예측 1 기준 정확도 = precision
# ---------------------------------------------------------

if pred_1_count > 0:
    pred_1_precision = pred_1_actual_1_count / pred_1_count
else:
    pred_1_precision = np.nan

print("예측 1 기준 정확도 precision:", pred_1_precision)

6. valid_df 생성
top feature 수: 27
lag 생성 후 valid_df shape: (1607, 56)
원본 base_feature 제거 후 valid_df shape: (1607, 29)
최종 컬럼 수: 29
target 생성 후 valid_df shape: (1607, 31)
최종 valid_df shape: (23, 31)
valid_base_df shape: (23, 34)
valid_df 기간:
2026-04-22 00:00:00 ~ 2026-05-22 00:00:00
7. Train 학습 후 valid_df 예측
사용 feature 수: 27
train_df shape: (1460, 30)
X_train shape: (1460, 27)
train target 분포:
target_5d_up_1pct
0    748
1    712
Name: count, dtype: int64
target_5d_up_1pct
0    0.512329
1    0.487671
Name: proportion, dtype: float64
valid_df shape: (23, 31)
예측 가능한 valid row 수: 23
예측 불가능 row 수: 0
모델 학습 완료
valid_df 예측 완료


,Date,SMH_adj_close,target_5d_up_1pct,pred_proba,pred
0,2026-04-22,476.829987,1.0,0.528,1
1,2026-04-23,481.850006,1.0,0.442,0
2,2026-04-24,506.440002,0.0,0.482,0
3,2026-04-27,506.260010,0.0,0.472,0
4,2026-04-28,491.209991,1.0,0.396,0
5,2026-04-29,499.579987,1.0,0.456,0
6,2026-04-30,506.720001,1.0,0.392,0
7,2026-05-01,509.820007,1.0,0.408,0
8,2026-05-04,506.790009,1.0,0.458,0
9,2026-05-05,522.690002,1.0,0.508,1


예측값 분포:
pred
0    13
1    10
Name: count, dtype: Int64
7. 예측 1 기준 정확도 확인
평가 가능 row 수: 18
예측 1 개수: 6
예측 1 중 실제 1 개수: 3
예측 1 기준 정확도 precision: 0.5
